# DataFrame: Combining through Concat and Merge

## Setup code

In [5]:
import pandas as pd

In [14]:
pop_url = "https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-population.csv"
df_pop_raw = pd.read_csv(pop_url)
df_pop_raw

,state/region,ages,year,population
0,AL,under18,2012,1117489.0
1,AL,total,2012,4817528.0
2,AL,under18,2010,1130966.0
3,AL,total,2010,4785570.0
4,AL,under18,2011,1125763.0
...,...,...,...,...
2539,USA,total,2010,309326295.0
2540,USA,under18,2011,73902222.0
2541,USA,total,2011,311582564.0
2542,USA,under18,2012,73708179.0


In [15]:
abbrev_url = "https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-abbrevs.csv"
df_abbrevs = pd.read_csv(abbrev_url)
df_abbrevs.head(5)

,state,abbreviation
0,Alabama,AL
1,Alaska,AK
2,Arizona,AZ
3,Arkansas,AR
4,California,CA


In [16]:
areas_url = "https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-areas.csv"
df_areas = pd.read_csv(areas_url)
df_areas.head(5)

,state,area (sq. mi)
0,Alabama,52423
1,Alaska,656425
2,Arizona,114006
3,Arkansas,53182
4,California,163707


In [9]:
# Split population data into two pieces for the exercise
pop_1990s = df_pop_raw[df_pop_raw['year'] < 2000]
pop_2000s = df_pop_raw[df_pop_raw['year'] >= 2000]

---

## 1. Concatenate Data (`pd.concat`)

You have two population datasets (`pop_1990s` and `pop_2000s`). Stack them vertically to create a single DataFrame named `pop_full`.

In [17]:
print(pop_1990s.shape)
print(pop_2000s.shape)

(1060, 4)
(1484, 4)


In [19]:
# 1. Concatenate Data
pop_full = pd.concat([pop_1990s, pop_2000s])
print(pop_full.shape)

(2544, 4)


## 2. First Join (`pd.merge` with differing column names)

Merge `pop_full` with `df_abbrevs` to add the full state names to the population data.

*Hint: The columns don't have the same name. You will need to match `state/region` from the population data with `abbreviation` from the abbreviations data using a left join.*

In [24]:
print(pop_full.shape, pop_full.columns.tolist())
print(df_abbrevs.shape, df_abbrevs.columns.tolist())

(2544, 4) ['state/region', 'ages', 'year', 'population']
(51, 2) ['state', 'abbreviation']


In [26]:
print(pop_full['state/region'].head(3))
print()
print(df_abbrevs['abbreviation'].head(3))

26    AL
27    AL
30    AL
Name: state/region, dtype: str

0    AL
1    AK
2    AZ
Name: abbreviation, dtype: str


In [32]:
# 2. First Join (differing column names)
# Using a left join ensures we don't lose population data if an abbreviation is missing
merged = pd.merge(
    pop_full,
    df_abbrevs,
    how='left',
    left_on='state/region',
    right_on='abbreviation'
)

In [33]:
# After the merge, the number of columns is the sum of the number of columns in the two DataFrames
print(merged.shape)
print(merged.shape[-1] == (pop_full.shape[-1] + df_abbrevs.shape[-1]))
merged.head(3)

(2544, 6)
True


,state/region,ages,year,population,state,abbreviation
0,AL,under18,1999,1121287.0,Alabama,AL
1,AL,total,1999,4430141.0,Alabama,AL
2,AL,total,1998,4404701.0,Alabama,AL


## 3. Drop Redundancy

Drop the `abbreviation` column from your merged DataFrame, as it is now a duplicate of the `state/region` column.

In [35]:
# 3. Drop Redundancy
merged = merged.drop('abbreviation', axis=1)
merged.head(3)

,state/region,ages,year,population,state
0,AL,under18,1999,1121287.0,Alabama
1,AL,total,1999,4430141.0,Alabama
2,AL,total,1998,4404701.0,Alabama


## 4. Second Join (`pd.merge` with matching column names)

Merge your new DataFrame with `df_areas` to add the area in square miles.
*Hint: Both tables now have a column explicitly named `state`, making this join more straightforward.*

In [36]:
# 4. Second Join (matching column names)
# Both tables now share the 'state' column
final_df = pd.merge(merged, df_areas, how='left', on='state')
final_df.head(3)

,state/region,ages,year,population,state,area (sq. mi)
0,AL,under18,1999,1121287.0,Alabama,52423.0
1,AL,total,1999,4430141.0,Alabama,52423.0
2,AL,total,1998,4404701.0,Alabama,52423.0
